# ChurnGuard Uplift V3 — Jupyter Notebook

Semi-synthetic uplift modeling experiment on the current ChurnGuard application dataset.

**Current application dataset:** `script/demo_prepare.rb` loads `db/seeds/synthetic_shop.rb` for the default shop `warsawstyle.myshopify.com` and then runs `ScoringEngine` + `UpliftDecisionEngine`.

This notebook uses the same dataset shape:

- `62` Shopify-style customers
- `292` paid orders in EUR
- `43` current application actions / exported demo actions
- customer-level RFM/churn features: recency, frequency, lifetime value, AOV, risk tier, risk score
- action-level fields: `treatment_key`, `uplift_score`, `expected_incremental_revenue`, `revenue_at_risk`, `outcome_window_days`

Important: the app currently has a **demo-ready uplift decision layer**, not a production-trained uplift model. Real production uplift needs randomized treatment/control assignment and observed outcomes. This notebook creates a **semi-synthetic treatment/control experiment** so the methodology can be shown and tested.

The notebook loads live-exported `current_app_*.csv` files by default.

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(exist_ok=True)

current_customers_path = DATA_DIR / "current_app_customers.csv"
current_orders_path = DATA_DIR / "current_app_orders.csv"
current_actions_path = DATA_DIR / "current_app_actions.csv"

if current_customers_path.exists() and current_orders_path.exists() and current_actions_path.exists():
    dataset_source = "current Rails app export"
    customers = pd.read_csv(current_customers_path)
    orders = pd.read_csv(current_orders_path)
    app_actions = pd.read_csv(current_actions_path)
else:
    dataset_source = "reproducible synthetic fallback"
    customers = pd.read_csv(DATA_DIR / "churnguard_synthetic_customers.csv")
    orders = pd.read_csv(DATA_DIR / "churnguard_synthetic_orders.csv")
    app_actions = pd.read_csv(DATA_DIR / "churnguard_uplift_v3_demo_actions.csv")

fallback_scores = {"high": 0.82, "medium": 0.58, "low": 0.25}
customers["risk_score"] = customers.apply(
    lambda r: fallback_scores.get(str(r.get("risk_tier", "low")), 0.25)
    if pd.isna(r.get("risk_score")) else float(r.get("risk_score")),
    axis=1,
)

print("dataset_source:", dataset_source)
print(customers.shape, orders.shape, app_actions.shape)
display(customers.head())
display(app_actions.head())

dataset_source: current Rails app export
(62, 10) (292, 6) (43, 11)


,shopify_customer_id,name,email,lifetime_value,days_since_last_order,frequency,aov,risk_tier,risk_score,last_scored_on
0,10001,Kamil Nowakowska,kamil.nowakowska56@interia.pl,49.0,489,1,49.0,high,0.82,2026-06-28
1,10002,Iwona Kamiński,iwona.kaminski88@gmail.com,89.0,444,1,89.0,high,0.82,2026-06-28
2,10003,Kamil Olszewski,kamil.olszewski73@wp.pl,293.0,414,2,146.5,high,0.82,2026-06-28
3,10004,Aleksandra Mazur,aleksandra.mazur80@o2.pl,55.0,399,1,55.0,high,0.82,2026-06-28
4,10005,Natalia Jaworska,natalia.jaworska33@onet.pl,263.0,384,2,131.5,high,0.82,2026-06-28


,shopify_customer_id,status,treatment_key,risk_tier,uplift_score,expected_incremental_revenue,revenue_at_risk,proposed_discount,model_version,holdout,outcome_window_days
0,10002,pending,winback_15,high,0.2315,17.51,22.25,15%,uplift-v3-demo-baseline,False,30
1,10003,pending,winback_15,high,0.2115,26.34,73.25,15%,uplift-v3-demo-baseline,False,30
2,10004,pending,winback_15,high,0.2315,10.82,13.75,15%,uplift-v3-demo-baseline,False,30
3,10005,pending,winback_15,high,0.2115,23.64,65.75,15%,uplift-v3-demo-baseline,False,30
4,10006,pending,winback_15,high,0.2315,39.36,50.00,15%,uplift-v3-demo-baseline,False,30


## 1. What dataset is this?

The Rails application demo uses `script/demo_prepare.rb`.

That script:

1. loads `db/seeds/synthetic_shop.rb` for `warsawstyle.myshopify.com`;
2. creates a realistic Polish Shopify-style fashion store;
3. creates customers and paid orders;
4. runs `ScoringEngine` to assign RFM risk tiers;
5. runs `UpliftDecisionEngine` to create pending retention actions.

The dataset is not a production customer dataset. It is a deterministic synthetic showroom dataset for demos, tests and diploma explanation.

In [2]:
summary = pd.DataFrame({
    "metric": [
        "customers",
        "orders",
        "demo uplift actions",
        "total LTV",
        "median LTV",
        "avg orders/customer",
    ],
    "value": [
        len(customers),
        len(orders),
        len(app_actions),
        round(customers["lifetime_value"].sum(), 2),
        round(customers["lifetime_value"].median(), 2),
        round(customers["frequency"].mean(), 2),
    ],
})
display(summary)
display(customers["risk_tier"].value_counts().rename_axis("risk_tier").reset_index(name="customers"))
display(app_actions["treatment_key"].value_counts().rename_axis("treatment").reset_index(name="actions"))

,metric,value
0,customers,62.00
1,orders,292.00
2,demo uplift actions,43.00
3,total LTV,199331.00
4,median LTV,1524.00
5,avg orders/customer,4.71


,risk_tier,customers
0,low,26
1,medium,19
2,high,17


,treatment,actions
0,nudge_10,21
1,winback_15,19
2,vip_personal_offer,3


## 2. Recreate the current app decision layer

The current application uses `UpliftDecisionEngine` as a deterministic demo decision layer. It estimates:

- best treatment;
- uplift score;
- expected incremental revenue;
- approval-card fields.

This is useful for product demo and diploma defense, but it is not yet trained from randomized experimental data.

In [3]:
def app_treatment_for(row, median_ltv):
    if row["lifetime_value"] >= median_ltv * 1.4 and row["frequency"] >= 3:
        return "vip_personal_offer"
    if row["days_since_last_order"] >= 150 or row["risk_tier"] == "high":
        return "winback_15"
    return "nudge_10"

def app_uplift_score(row, treatment_key, median_ltv):
    recency_component = min(max(row["days_since_last_order"] / 240.0, 0.0), 1.0) * 0.08
    frequency_component = 0.045 if row["frequency"] <= 1 else 0.025
    risk_component = row["risk_score"] * 0.075
    value_component = 0.025 if row["lifetime_value"] >= median_ltv else 0.01
    treatment_component = {
        "winback_15": 0.035,
        "vip_personal_offer": 0.03,
        "nudge_10": 0.02,
    }[treatment_key]
    return round(min(max(recency_component + frequency_component + risk_component + value_component + treatment_component, 0.02), 0.32), 4)

def expected_incremental_revenue(row, uplift, treatment_key):
    cost_rate = {"winback_15": 0.15, "nudge_10": 0.10, "vip_personal_offer": 0.12}[treatment_key]
    return round(uplift * row["aov"] * (1 - cost_rate), 2)

median_ltv = customers["lifetime_value"].median()
eligible = customers[customers["risk_tier"].isin(["high", "medium"])].copy()
eligible["app_treatment_key"] = eligible.apply(lambda r: app_treatment_for(r, median_ltv), axis=1)
eligible["app_uplift_score"] = eligible.apply(lambda r: app_uplift_score(r, r["app_treatment_key"], median_ltv), axis=1)
eligible["app_expected_incremental_revenue"] = eligible.apply(
    lambda r: expected_incremental_revenue(r, r["app_uplift_score"], r["app_treatment_key"]),
    axis=1,
)
eligible = eligible[eligible["app_expected_incremental_revenue"] >= 10].copy()

display(eligible[[
    "shopify_customer_id", "name", "risk_tier", "days_since_last_order", "frequency",
    "lifetime_value", "app_treatment_key", "app_uplift_score", "app_expected_incremental_revenue"
]].head(10))
print("Eligible demo approval actions:", len(eligible))

,shopify_customer_id,name,risk_tier,days_since_last_order,frequency,lifetime_value,app_treatment_key,app_uplift_score,app_expected_incremental_revenue
1,10002,Iwona Kamiński,high,444,1,89.0,winback_15,0.2315,17.51
2,10003,Kamil Olszewski,high,414,2,293.0,winback_15,0.2115,26.34
3,10004,Aleksandra Mazur,high,399,1,55.0,winback_15,0.2315,10.82
4,10005,Natalia Jaworska,high,384,2,263.0,winback_15,0.2115,23.64
5,10006,Marcin Kozłowski,high,369,1,200.0,winback_15,0.2315,39.36
6,10007,Michał Jaworska,high,354,2,357.0,winback_15,0.2115,32.09
7,10008,Wojciech Mazur,high,339,1,75.0,winback_15,0.2315,14.76
8,10009,Marcin Górska,high,331,1,89.0,winback_15,0.2315,17.51
9,10010,Łukasz Grabowska,high,324,2,242.0,winback_15,0.2115,21.75
10,10011,Rafał Krawczyk,high,319,2,290.0,winback_15,0.2115,26.07


Eligible demo approval actions: 35


## 3. Build a semi-synthetic treatment/control experiment

Because the current app dataset has no real randomized treatment/control outcomes, we simulate an experiment:

- `treatment = 1`: customer receives the app-recommended retention action;
- `treatment = 0`: customer is held out;
- `outcome = 1`: customer makes a repeat purchase in the next 30 days.

The simulation intentionally uses the app features, but adds randomness. This lets us train and evaluate uplift models while keeping the honest limitation clear.

In [4]:
rng = np.random.default_rng(20260928)
df = customers.copy()
df["recommended_treatment"] = df.apply(lambda r: app_treatment_for(r, median_ltv), axis=1)
df["true_uplift"] = df.apply(lambda r: app_uplift_score(r, r["recommended_treatment"], median_ltv), axis=1)

# Baseline purchase probability without treatment.
df["base_purchase_prob"] = (
    0.36
    - 0.0009 * df["days_since_last_order"]
    + 0.018 * np.log1p(df["frequency"])
    + 0.000025 * df["lifetime_value"]
    - 0.10 * (df["risk_tier"] == "high").astype(float)
)
df["base_purchase_prob"] = df["base_purchase_prob"].clip(0.03, 0.65)

# Randomized holdout. In production this must be assigned and logged by the app.
df["treatment"] = rng.binomial(1, 0.55, size=len(df))
df["outcome_prob"] = (df["base_purchase_prob"] + df["treatment"] * df["true_uplift"]).clip(0.01, 0.95)
df["repeat_purchase_30d"] = rng.binomial(1, df["outcome_prob"])
df["simulated_revenue_30d"] = np.where(
    df["repeat_purchase_30d"] == 1,
    np.maximum(df["aov"], 25),
    0.0,
)

display(df[[
    "shopify_customer_id", "risk_tier", "recommended_treatment", "treatment",
    "base_purchase_prob", "true_uplift", "outcome_prob", "repeat_purchase_30d"
]].head(12))
display(pd.crosstab(df["treatment"], df["repeat_purchase_30d"], margins=True))

,shopify_customer_id,risk_tier,recommended_treatment,treatment,base_purchase_prob,true_uplift,outcome_prob,repeat_purchase_30d
0,10001,high,winback_15,1,0.03,0.2315,0.2615,0
1,10002,high,winback_15,1,0.03,0.2315,0.2615,0
2,10003,high,winback_15,1,0.03,0.2115,0.2415,1
3,10004,high,winback_15,1,0.03,0.2315,0.2615,0
4,10005,high,winback_15,0,0.03,0.2115,0.0300,0
5,10006,high,winback_15,1,0.03,0.2315,0.2615,0
6,10007,high,winback_15,1,0.03,0.2115,0.2415,0
7,10008,high,winback_15,0,0.03,0.2315,0.0300,0
8,10009,high,winback_15,1,0.03,0.2315,0.2615,1
9,10010,high,winback_15,1,0.03,0.2115,0.2415,1


repeat_purchase_30d,0,1,All
treatment,,,
0,25,4,29
1,18,15,33
All,43,19,62


## 4. Prepare model features

For an uplift model, the same customer features are used in both treatment and control groups.

We include:

- recency;
- frequency;
- lifetime value;
- AOV;
- risk score;
- risk tier one-hot flags.

In [5]:
feature_cols = [
    "days_since_last_order",
    "frequency",
    "lifetime_value",
    "aov",
    "risk_score",
]
X_base = df[feature_cols].astype(float).copy()
for tier in ["high", "medium", "low"]:
    X_base[f"risk_{tier}"] = (df["risk_tier"] == tier).astype(float)

def standardize(frame):
    mu = frame.mean()
    sigma = frame.std(ddof=0).replace(0, 1)
    return (frame - mu) / sigma, mu, sigma

X_std, X_mu, X_sigma = standardize(X_base)
y = df["repeat_purchase_30d"].astype(float).values
t = df["treatment"].astype(float).values

print(X_std.shape)
display(X_std.head())

(62, 8)


,days_since_last_order,frequency,lifetime_value,aov,risk_score,risk_high,risk_medium,risk_low
0,2.984908,-1.187778,-0.720958,-1.222469,1.31862,1.626978,-0.664726,-0.849837
1,2.482047,-1.187778,-0.711849,-1.107928,1.31862,1.626978,-0.664726,-0.849837
2,2.146805,-0.867595,-0.665395,-0.943275,1.31862,1.626978,-0.664726,-0.849837
3,1.979185,-1.187778,-0.719592,-1.205288,1.31862,1.626978,-0.664726,-0.849837
4,1.811564,-0.867595,-0.672226,-0.986228,1.31862,1.626978,-0.664726,-0.849837


## 5. Minimal logistic regression in NumPy

This notebook avoids hidden library magic. The logistic regression below is enough for a small teaching/demo dataset.

In [7]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -40, 40)))

def fit_logistic(X, y, lr=0.08, epochs=4000, l2=0.01):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    X_design = np.c_[np.ones(len(X)), X]
    w = np.zeros(X_design.shape[1])
    for _ in range(epochs):
        p = sigmoid(X_design @ w)
        grad = (X_design.T @ (p - y)) / len(y)
        grad[1:] += l2 * w[1:]
        w -= lr * grad
    return w

def predict_logistic(w, X):
    X = np.asarray(X, dtype=float)
    X_design = np.c_[np.ones(len(X)), X]
    return sigmoid(X_design @ w)

## 6. S-Learner

S-Learner trains one outcome model with treatment included as a feature.

To estimate uplift:

1. predict outcome if `treatment=1`;
2. predict outcome if `treatment=0`;
3. subtract the two predictions.

In [8]:
X_s = X_std.copy()
X_s["treatment"] = t
for col in X_std.columns:
    X_s[f"{col}_x_treatment"] = X_std[col] * t

w_s = fit_logistic(X_s.values, y, lr=0.06, epochs=5000, l2=0.02)

X_s_t1 = X_std.copy()
X_s_t1["treatment"] = 1.0
for col in X_std.columns:
    X_s_t1[f"{col}_x_treatment"] = X_std[col] * 1.0

X_s_t0 = X_std.copy()
X_s_t0["treatment"] = 0.0
for col in X_std.columns:
    X_s_t0[f"{col}_x_treatment"] = 0.0

df["s_learner_p_treat"] = predict_logistic(w_s, X_s_t1.values)
df["s_learner_p_control"] = predict_logistic(w_s, X_s_t0.values)
df["s_learner_uplift"] = df["s_learner_p_treat"] - df["s_learner_p_control"]
display(df[["shopify_customer_id", "risk_tier", "true_uplift", "s_learner_uplift"]].head())

,shopify_customer_id,risk_tier,true_uplift,s_learner_uplift
0,10001,high,0.2315,0.083644
1,10002,high,0.2315,0.065431
2,10003,high,0.2115,0.098640
3,10004,high,0.2315,0.055251
4,10005,high,0.2115,0.090149


## 7. T-Learner

T-Learner trains two separate outcome models:

- one model on treated customers;
- one model on control customers.

Uplift is the difference between both predictions.

In [9]:
treated_mask = t == 1
control_mask = t == 0

w_treat = fit_logistic(X_std.loc[treated_mask].values, y[treated_mask], lr=0.06, epochs=5000, l2=0.02)
w_control = fit_logistic(X_std.loc[control_mask].values, y[control_mask], lr=0.06, epochs=5000, l2=0.02)

df["t_learner_p_treat"] = predict_logistic(w_treat, X_std.values)
df["t_learner_p_control"] = predict_logistic(w_control, X_std.values)
df["t_learner_uplift"] = df["t_learner_p_treat"] - df["t_learner_p_control"]

display(df[["shopify_customer_id", "risk_tier", "true_uplift", "t_learner_uplift"]].head())

,shopify_customer_id,risk_tier,true_uplift,t_learner_uplift
0,10001,high,0.2315,0.277582
1,10002,high,0.2315,0.232027
2,10003,high,0.2115,0.257793
3,10004,high,0.2315,0.194681
4,10005,high,0.2115,0.225936


## 8. Uplift evaluation: Qini, AUUC, Lift@K

For production, the most important metric is real incremental revenue in treatment vs holdout. Offline, uplift models are commonly evaluated with uplift curves, Qini-style cumulative gain and Lift@K.

In [10]:
def uplift_curve(frame, score_col, outcome_col="repeat_purchase_30d", treatment_col="treatment"):
    ranked = frame.sort_values(score_col, ascending=False).reset_index(drop=True).copy()
    ranked["treated"] = ranked[treatment_col] == 1
    ranked["control"] = ranked[treatment_col] == 0
    ranked["treated_outcome"] = ranked[outcome_col] * ranked["treated"]
    ranked["control_outcome"] = ranked[outcome_col] * ranked["control"]
    ranked["cum_treated"] = ranked["treated"].cumsum().replace(0, np.nan)
    ranked["cum_control"] = ranked["control"].cumsum().replace(0, np.nan)
    ranked["cum_treated_outcome"] = ranked["treated_outcome"].cumsum()
    ranked["cum_control_outcome"] = ranked["control_outcome"].cumsum()
    ranked["uplift_gain"] = (
        ranked["cum_treated_outcome"]
        - ranked["cum_treated"] * ranked["cum_control_outcome"] / ranked["cum_control"]
    ).fillna(0)
    ranked["population_fraction"] = (np.arange(len(ranked)) + 1) / len(ranked)
    return ranked

def auuc(curve):
    return float(np.trapz(curve["uplift_gain"], curve["population_fraction"]))

def lift_at_k(frame, score_col, k=0.3):
    n = max(1, int(len(frame) * k))
    top = frame.sort_values(score_col, ascending=False).head(n)
    treat = top[top["treatment"] == 1]
    control = top[top["treatment"] == 0]
    if len(treat) == 0 or len(control) == 0:
        return np.nan
    return treat["repeat_purchase_30d"].mean() - control["repeat_purchase_30d"].mean()

metrics = []
for name, col in [
    ("Current app demo score", "true_uplift"),
    ("S-Learner", "s_learner_uplift"),
    ("T-Learner", "t_learner_uplift"),
]:
    curve = uplift_curve(df, col)
    metrics.append({
        "model": name,
        "AUUC": round(auuc(curve), 4),
        "Qini_final_gain": round(float(curve["uplift_gain"].iloc[-1]), 4),
        "Lift@30%": round(float(lift_at_k(df, col, 0.30)), 4),
    })

metrics_df = pd.DataFrame(metrics)
display(metrics_df)

/var/folders/fb/cwxyyxld545_nsbqqq7tj4f40000gp/T/ipykernel_8753/994048219.py:19: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(curve["uplift_gain"], curve["population_fraction"]))
/var/folders/fb/cwxyyxld545_nsbqqq7tj4f40000gp/T/ipykernel_8753/994048219.py:19: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(curve["uplift_gain"], curve["population_fraction"]))
/var/folders/fb/cwxyyxld545_nsbqqq7tj4f40000gp/T/ipykernel_8753/994048219.py:19: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(curve["uplift_gain"], curve["population_fraction"]))


,model,AUUC,Qini_final_gain,Lift@30%
0,Current app demo score,3.8747,10.4483,0.1299
1,S-Learner,7.1327,10.4483,0.7750
2,T-Learner,7.3565,10.4483,0.7750


## 9. Export predictions for ChurnGuard approval queue

The app does not need to show every model detail. It needs fields that map to approval cards:

- customer;
- best action;
- uplift score;
- expected incremental revenue;
- holdout flag;
- outcome window.

In [11]:
pred = df.copy()
pred["model_uplift_score"] = pred["t_learner_uplift"].clip(0, 1)
pred["treatment_key"] = pred["recommended_treatment"]
pred["expected_incremental_revenue"] = pred.apply(
    lambda r: expected_incremental_revenue(
        {
            "aov": r["aov"],
        },
        r["model_uplift_score"],
        r["treatment_key"],
    ),
    axis=1,
)
pred["holdout"] = pred["treatment"] == 0
pred["outcome_window_days"] = 30

export_cols = [
    "shopify_customer_id",
    "name",
    "risk_tier",
    "risk_score",
    "treatment_key",
    "model_uplift_score",
    "expected_incremental_revenue",
    "holdout",
    "outcome_window_days",
]
export = pred[export_cols].sort_values("expected_incremental_revenue", ascending=False)
export_path = EXPORT_DIR / "uplift_v3_predictions_for_churnguard.csv"
export.to_csv(export_path, index=False)
print(export_path)
display(export.head(12))

exports/uplift_v3_predictions_for_churnguard.csv


,shopify_customer_id,name,risk_tier,risk_score,treatment_key,model_uplift_score,expected_incremental_revenue,holdout,outcome_window_days
55,10056,Tomasz Woźniak,low,0.25,vip_personal_offer,0.860095,1392.67,True,30
60,10061,Robert Woźniak,low,0.25,vip_personal_offer,0.840971,1062.30,False,30
54,10055,Robert Kubiak,low,0.25,vip_personal_offer,0.739614,752.39,True,30
61,10062,Piotr Michalska,low,0.25,vip_personal_offer,0.726052,722.22,False,30
52,10053,Marcin Majewski,low,0.25,vip_personal_offer,0.668273,535.56,True,30
53,10054,Rafał Zawadzki,low,0.25,vip_personal_offer,0.627298,527.06,False,30
59,10060,Katarzyna Woźniak,low,0.25,vip_personal_offer,0.611991,494.81,False,30
46,10047,Jakub Nowak,low,0.25,vip_personal_offer,0.629448,443.38,True,30
57,10058,Tomasz Górska,low,0.25,vip_personal_offer,0.543756,384.66,True,30
40,10041,Paulina Zajac,low,0.25,vip_personal_offer,0.590989,373.28,True,30


## 10. What to say on defense

Use this wording:

> The current application dataset is a deterministic synthetic Shopify-style dataset generated by `db/seeds/synthetic_shop.rb` and prepared by `script/demo_prepare.rb`. It contains customers, orders, RFM risk tiers and Uplift V3 demo actions. It is enough to demonstrate the product workflow and decision layer.

> For a production uplift model, this dataset is not enough because it does not contain randomized treatment/control outcomes. Therefore, the notebook creates a semi-synthetic uplift experiment on top of the current dataset and shows how S-Learner/T-Learner, AUUC, Qini and Lift@K would work. The production roadmap is to collect real holdout/outcome data from Shopify/Klaviyo pilots.